# 06 - Uplift Modelling & Heterogeneous Treatment Effects
**Milestone 6 (EconML)**

Average effects hide the fact that a campaign helps some customers and harms
others. We estimate the **conditional average treatment effect (CATE)** with
EconML's `CausalForestDML` (T-learner fallback), then evaluate targeting with a
**Qini curve** and **AUUC** on the Hillstrom e-mail experiment and the IHDP
semi-synthetic benchmark.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def _find_project_root(start):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, "src")) and os.path.exists(
            os.path.join(d, "src", "config.py")
        ):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.path.abspath(start)
        d = parent

PROJECT_ROOT = _find_project_root(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import artifact_path, raw_path, ensure_directories
ensure_directories()
FIGS = artifact_path("figures")
os.makedirs(FIGS, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("FIGURES    :", FIGS)


## 1. Hillstrom e-mail campaign - who should we target?
Treatment = received the *Mens E-Mail* (vs. no e-mail); outcome = visit.

In [ ]:
from src.data_pipeline import datasets
from src.core.causal import uplift as up

hills = datasets.load_hillstrom()
hills = hills[hills.segment.isin(["Mens E-Mail", "No E-Mail"])].copy()
hills["treatment"] = (hills.segment == "Mens E-Mail").astype(int)
features = ["recency", "history", "mens", "womens", "newbie"]
print(hills.shape, "| treated share:", round(hills.treatment.mean(), 3))
hills[["treatment", "visit", "conversion", "spend"]].describe().loc[["mean", "std"]]

In [ ]:
hill_res = up.estimate_uplift(hills, treatment="treatment", outcome="visit",
                                features=features, learner="causal_forest", n_estimators=200)
{ "engine": hill_res["engine"], "ate": round(hill_res["ate"], 4),
  "cate_positive_share": round(hill_res["cate_positive_share"], 3),
  "auuc": round(hill_res["auuc"], 1), "qini@10%": round(hill_res["qini_at_10pct"], 2) }

### Qini curve - targeting the highest-uplift customers beats random targeting

In [ ]:
q = hill_res["qini"]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(q["fraction"], q["qini"], color="#4f46e5", lw=2, label="model ranking")
ax.plot(q["fraction"], q["random_baseline"], "--", color="#94a3b8", label="random targeting")
ax.set_xlabel("fraction of population targeted"); ax.set_ylabel("incremental visits (Qini)")
ax.set_title(f"Qini curve - AUUC = {hill_res['auuc']:.0f}"); ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(FIGS, "06_qini_hillstrom.png"), dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
pd.DataFrame({
    "feature": list(hill_res["feature_importance"].keys()),
    "importance": list(hill_res["feature_importance"].values()),
}).sort_values("importance", ascending=False)

### Segment profile - who are the high-uplift customers?

In [ ]:
pd.DataFrame({
    "top_uplift_segment": hill_res["top_segment_profile"],
    "bottom_uplift_segment": hill_res["bottom_segment_profile"],
})

## 2. IHDP semi-synthetic benchmark (known ground truth)
The IHDP simulator encodes a non-linear response surface, so a good uplift model must recover real heterogeneity.

In [ ]:
ihdp = datasets.load_ihdp()
ihdp_feats = [c for c in ihdp.columns if c.startswith("x")][:10]
ihdp_res = up.estimate_uplift(ihdp, treatment="treatment", outcome="y_factual",
                              features=ihdp_feats, learner="causal_forest", n_estimators=150)
{ "engine": ihdp_res["engine"], "ate": round(ihdp_res["ate"], 3),
  "cate_std": round(ihdp_res["cate_std"], 3),
  "cate_positive_share": round(ihdp_res["cate_positive_share"], 3),
  "auuc": round(ihdp_res["auuc"], 2) }

In [ ]:
q = ihdp_res["qini"]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(q["fraction"], q["qini"], color="#0ea5e9", lw=2, label="model ranking")
ax.plot(q["fraction"], q["random_baseline"], "--", color="#94a3b8", label="random targeting")
ax.set_xlabel("fraction targeted"); ax.set_ylabel("Qini"); ax.set_title("Qini curve - IHDP")
ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(FIGS, "06_qini_ihdp.png"), dpi=120, bbox_inches="tight"); plt.show()

## Takeaways
* A positive AUUC means the model's ranking beats random targeting - we can raise incremental conversions with the same budget.
* `cate_positive_share` tells you what fraction of the population benefits; a "persuadable" segment can be isolated for campaigns and the rest suppressed.
* EconML's `CausalForestDML` doubles as a confounder adjustment (via the DML nuisance models), so heterogeneity estimates are debiased.